# See it

**Rung 1 of 5 (next: rung 2, [04, Keep what you need](04_keep_what_you_need.ipynb)). You are
here: the summary is too long to read, so draw it.**
Trigger: the summary is too long to read.

`trace.draw()` turns a trace into a picture of the dataflow: one box per operation, framed by the
modules they ran in. This notebook is the dials on that one method: collapse repeated modules,
focus on one, roll a loop up, colour and size the boxes by data, show a dict return as a node,
and the one-call form.

> **You'll want to know:** [`tl.trace`](01_capture_and_look.ipynb) (one call records a forward
> pass), [labels and module addresses](02_address_and_read.ipynb) (`relu_1_2`, `blocks.2`), and
> [the summary](01_capture_and_look.ipynb) (the table this picture replaces).

In [ ]:
import tempfile
from pathlib import Path

import torch
from IPython.display import display

import torchlens as tl

PICTURES = Path(
    tempfile.mkdtemp(prefix="torchlens_learn_")
)  # drawings land here, not beside the notebook

In [ ]:
from learn_models import tiny_cnn, tiny_dict_model, tiny_mlp, tiny_rnn  # tiny seeded CPU models

cnn, image = tiny_cnn()  # a stem, six conv blocks (conv, BatchNorm, ReLU), pooling, a linear head

The six-block CNN with `collapse="max"`: the five identical blocks fold into one summary node,
and the last block, whose output feeds the head, keeps its detail.


In [ ]:
big = tl.trace(cnn, image)
big.draw(path=str(PICTURES / "cnn_max"), file_format="svg", collapse="max")

## 1. The plain picture

With no dials, every operation is a box and every module a frame; on this CNN that is a tall
picture, the one full graph in this notebook, and the dials below are how you shorten it. `draw` writes the
picture to the path stem you give it, in the `file_format` you name, and returns a `Drawing`: it
displays inline when it is the last thing in a cell, keeps the DOT source in `.dot`, and
`.save(path)` writes it again elsewhere. In a notebook, give `draw` a path stem so nothing lands
beside the notebook; `PICTURES` here is a temporary directory.

In [ ]:
drawing = big.draw(path=str(PICTURES / "cnn"), file_format="svg")
big.draw(path=str(PICTURES / "cnn"), file_format="pdf")  # the same picture as a PDF
print(len(big.ops), "ops |", sorted(p.name for p in PICTURES.glob("cnn.*")), "written")
print("DOT source begins:", drawing.dot[:70].replace("\n", " "))
drawing

## 2. Collapse

`collapse=` folds modules into single nodes without touching the trace. `"max"` folds as far
as it can: identical siblings become one summary node (`blocks.0-4` above), and a sibling that
differs stays open. `"auto"` folds only when the picture is big enough to need it, and a float
between 0 and 1 picks a point in between. `module_collapse_order` ranks the modules by how much
each would hide, so you can see which fold first.


In [ ]:
print("folds first:", big.module_collapse_order[:4])

## 3. Focus and depth

`module=` draws one module and what feeds it; `depth=` limits how far into nested modules the
frames go, so `depth=1` shows the top-level modules as single nodes.

In [ ]:
display(big.draw(path=str(PICTURES / "block2"), file_format="svg", module="blocks.2"))
display(big.draw(path=str(PICTURES / "depth1"), file_format="svg", depth=1))

## 4. Rolled and unrolled

A loop in `forward` runs the same layer several passes. The default picture unrolls it, one node
per pass; `rolled=True` draws each layer once and writes the pass count on the node.

In [ ]:
rnn, seq = tiny_rnn()  # one linear cell and a tanh, applied three times in a Python loop
loop = tl.trace(rnn, seq)
display(loop.draw(path=str(PICTURES / "rnn_rolled"), file_format="svg", rolled=True))
display(loop.draw(path=str(PICTURES / "rnn_unrolled"), file_format="svg"))

## 5. Colour and size as data

Any numeric field of a record can drive a visual channel. `color_by=` fills the boxes along a
sequential ramp and `size_by=` scales them; `"dims"` sizes each box by its activation's size
per example (the batch axis left out), on a bounded scale. The legend states the mapping.
`theme=` swaps the palette: `"paper"`, `"dark"`, `"colorblind"` or `"high_contrast"`. The
two-layer MLP is small enough to read both channels at a glance.


In [ ]:
mlp, mx = tiny_mlp()  # two linear layers with a ReLU between them
small = tl.trace(mlp, mx)
display(
    small.draw(
        path=str(PICTURES / "by_flops"),
        file_format="svg",
        color_by="flops_forward",
    )
)
display(
    small.draw(
        path=str(PICTURES / "by_dims"),
        file_format="svg",
        size_by="dims",
        theme="dark",
    )
)

## 6. Containers in the picture

A model that returns a dict draws its leaves as ordinary output nodes by default.
`show_containers="nodes"` adds the dict itself as a node, with `logits` and `hidden` hanging off
it, so the return structure is visible in the graph.

In [ ]:
dict_model, dx = tiny_dict_model()  # returns {"logits": ..., "hidden": ...}
dt = tl.trace(dict_model, dx)
display(dt.draw(path=str(PICTURES / "dict_default"), file_format="svg"))
display(
    dt.draw(
        path=str(PICTURES / "dict_nodes"),
        file_format="svg",
        show_containers="nodes",
    )
)

## 7. One call

When the picture is all you want, chain the two calls: `tl.trace(model, x).draw(...)`. The dials
above all pass through, and the capture takes the same input forms as always, so `input_shape=`
works when you have no input to hand.

In [ ]:
tl.trace(cnn, input_shape=(1, 1, 16, 16)).draw(
    path=str(PICTURES / "one_call"), file_format="svg", collapse="max"
)

## 8. Where this leaves you

One method, a handful of dials: `collapse`, `module`, `depth`, `rolled`, `color_by`, `size_by`,
`theme`, `show_containers`, and a chained one-call form. The backward graph has its own method,
`trace.draw(graph="backward")`, in [06, Gradients](06_gradients.ipynb).

**One thing about networks:** the collapse works because five of the six blocks are structurally
identical, and the one that stays open is the one whose output leaves the stack. Repetition is
the architecture's main idea, and the folded picture shows depth as repetition rather than as a
list of layers.

**Go deeper:** next rung, [04, Keep what you need](04_keep_what_you_need.ipynb): choose what the
capture saves. Recipe card: [`visualize.ipynb`](../../examples/5min/visualize.ipynb). Reference:
[collapse](../../docs/reference/collapse.md) and
[encoding channels](../../docs/reference/encoding.md), and the glossary entries
[Collapse and color_by](../../docs/reference/glossary.md#capture-honesty-and-visualization).